In [1]:
import os
import numpy as np
import pandas as pd
print("Libraries imported Succesfully!")

Libraries imported Succesfully!


In [4]:
print("File Path Located!")
print(os.getcwd())

File Path Located!
C:\Users\Admin\Desktop\Ferry Capacity Utilization


In [6]:
df = pd.read_csv(r"C:\Users\Admin\Desktop\Ferry Capacity Utilization\Toronto Island Ferry Tickets.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 261538 entries, 0 to 261537
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   _id               261538 non-null  int64
 1   Timestamp         261538 non-null  str  
 2   Redemption Count  261538 non-null  int64
 3   Sales Count       261538 non-null  int64
dtypes: int64(3), str(1)
memory usage: 12.7 MB


In [7]:

print("--- 1. Data Ingestion & Structuring ---")

# Load the historical ticket dataset (2015-2025)
df = pd.read_csv('Toronto Island Ferry Tickets.csv')

# Display basic info to check columns and shape
print("Dataset Shape:", df.shape)
print("First 3 rows of raw data:")
print(df.head(3))

# Convert 'Timestamp' column to standard pandas datetime objects
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Sort the dataset chronologically by timestamp
df = df.sort_values('Timestamp').reset_index(drop=True)


print("\n--- 2. Multi-Resolution Data Aggregation ---")

# Set Timestamp as the index to make resampling easy
df_indexed = df.set_index('Timestamp')

# A. 15-Minute Resolution (Base dataset is already at 15-min intervals)
df_15min = df.copy()
print("15-Minute resolution records count:", len(df_15min))

# B. Hourly Resolution (Summing sales and redemptions per hour)
df_hourly = df_indexed.resample('h')[['Sales Count', 'Redemption Count']].sum().reset_index()
print("Hourly resolution records count:", len(df_hourly))

# C. Daily Resolution (Averaging or summing daily performance)
df_daily = df_indexed.resample('D')[['Sales Count', 'Redemption Count']].mean().reset_index()
print("Daily resolution records count:", len(df_daily))


--- 1. Data Ingestion & Structuring ---
Dataset Shape: (261538, 4)
First 3 rows of raw data:
   _id            Timestamp  Redemption Count  Sales Count
0    1  2025-12-21T22:30:00                14           16
1    2  2025-12-21T22:15:00                 1            0
2    3  2025-12-21T22:00:00                 2            0

--- 2. Multi-Resolution Data Aggregation ---
15-Minute resolution records count: 261538
Hourly resolution records count: 93298
Daily resolution records count: 3888


In [8]:

print("\n--- 1. Identify Missing or Irregular Intervals ---")

# Check for missing values in any column
missing_counts = df.isnull().sum()
print("Missing values per column:\n", missing_counts)

# Check time gaps between consecutive records to find irregular intervals
df['Time_Gap'] = df['Timestamp'].diff()
# Expected interval is usually 15 minutes
irregular_intervals = df[df['Time_Gap'] > pd.Timedelta(minutes=15)]
print(f"Total irregular/gapped time intervals found: {len(irregular_intervals):,}")


print("\n--- 2. Validate Negative or Zero-Count Anomalies ---")

# Check if there are any negative values in sales or redemptions
negative_sales = (df['Sales Count'] < 0).sum()
negative_redemptions = (df['Redemption Count'] < 0).sum()

print(f"Negative values in Sales Count: {negative_sales}")
print(f"Negative values in Redemption Count: {negative_redemptions}")

# Clean / clip anomalies to ensure values don't drop below zero
df['Sales Count'] = df['Sales Count'].clip(lower=0)
df['Redemption Count'] = df['Redemption Count'].clip(lower=0)
print("Anomaly validation complete: Negative counts successfully clipped to 0.")


print("\n--- 3. Smooth Extreme Spikes Using Rolling Statistics ---")

# Calculate Total Activity Load first
df['Total Activity Load'] = df['Sales Count'] + df['Redemption Count']

# Apply a rolling window (e.g., 4 periods = 1 hour window based on 15-min intervals)
# We use rolling mean to smooth out sudden erratic ticket spikes
window_size = 4 
df['Activity_Smoothed_Rolling_Mean'] = df['Total Activity Load'].rolling(window=window_size, min_periods=1).mean()

# Alternatively, apply rolling median to be robust against extreme outlier spikes
df['Activity_Smoothed_Rolling_Median'] = df['Total Activity Load'].rolling(window=window_size, min_periods=1).median()

print("Extreme spikes smoothed successfully using rolling statistics!")
print("\nSample of data after quality checks and smoothing:")
print(df[['Timestamp', 'Sales Count', 'Total Activity Load', 'Activity_Smoothed_Rolling_Mean', 'Activity_Smoothed_Rolling_Median']].head(10))


--- 1. Identify Missing or Irregular Intervals ---
Missing values per column:
 _id                 0
Timestamp           0
Redemption Count    0
Sales Count         0
dtype: int64
Total irregular/gapped time intervals found: 25,275

--- 2. Validate Negative or Zero-Count Anomalies ---
Negative values in Sales Count: 0
Negative values in Redemption Count: 0
Anomaly validation complete: Negative counts successfully clipped to 0.

--- 3. Smooth Extreme Spikes Using Rolling Statistics ---
Extreme spikes smoothed successfully using rolling statistics!

Sample of data after quality checks and smoothing:
            Timestamp  Sales Count  Total Activity Load  \
0 2015-05-01 13:30:00            1                    1   
1 2015-05-01 15:15:00            2                    2   
2 2015-05-01 15:45:00            1                    1   
3 2015-05-01 16:00:00            0                    1   
4 2015-05-04 16:00:00            2                    2   
5 2015-05-04 16:15:00            2      

In [9]:
print("\n--- 3. Data Quality & Consistency Checks ---")

# Check for any missing values across columns
print("Missing values check:\n", df.isnull().sum())

# Validate and clean negative or zero-count anomalies (clipping to ensure valid metrics)
df['Sales Count'] = df['Sales Count'].clip(lower=0)
df['Redemption Count'] = df['Redemption Count'].clip(lower=0)
print("Anomaly check completed: Negative values successfully cleaned.")


print("\n--- 4. Capacity Utilization Feature Engineering ---")

# 1. Total Activity Load (Combined sales and redemptions per interval)
df['Total Activity Load'] = df['Sales Count'] + df['Redemption Count']

# 2. Redemption Pressure Ratio (Redemptions relative to sales volume)
df['Redemption Pressure Ratio'] = df['Redemption Count'] / (df['Sales Count'] + 1)

# 3. Operational Load Index (OLI) - Normalized pressure score between 0 and 1
max_load = df['Total Activity Load'].max()
df['Operational Load Index'] = df['Total Activity Load'] / (max_load if max_load > 0 else 1)

# 4. Idle Capacity Indicator - Flags intervals with bottom 25% low activity load
idle_threshold = df['Total Activity Load'].quantile(0.25)
df['Idle Capacity Indicator'] = (df['Total Activity Load'] <= idle_threshold).astype(int)

# To ensure it captures sustained low activity, we check if the rolling average (over 2 periods) is below threshold
df['Sustained_Low_Activity'] = df['Total Activity Load'].rolling(window=2, min_periods=1).mean() <= idle_threshold
df['Idle Capacity Indicator'] = df['Sustained_Low_Activity'].astype(int)

# Drop temporary helper column
df = df.drop(columns=['Sustained_Low_Activity'])


print("\nFeature engineering completed successfully!")
print("\nSample of engineered feature dataset:")
print(df[['Timestamp', 'Sales Count', 'Redemption Count', 'Total Activity Load', 'Redemption Pressure Ratio', 'Operational Load Index', 'Idle Capacity Indicator']].head(10))




--- 3. Data Quality & Consistency Checks ---
Missing values check:
 _id                                 0
Timestamp                           0
Redemption Count                    0
Sales Count                         0
Time_Gap                            1
Total Activity Load                 0
Activity_Smoothed_Rolling_Mean      0
Activity_Smoothed_Rolling_Median    0
dtype: int64
Anomaly check completed: Negative values successfully cleaned.

--- 4. Capacity Utilization Feature Engineering ---

Feature engineering completed successfully!

Sample of engineered feature dataset:
            Timestamp  Sales Count  Redemption Count  Total Activity Load  \
0 2015-05-01 13:30:00            1                 0                    1   
1 2015-05-01 15:15:00            2                 0                    2   
2 2015-05-01 15:45:00            1                 0                    1   
3 2015-05-01 16:00:00            0                 1                    1   
4 2015-05-04 16:00:00        

In [10]:
print("\n--- 2. Temporal Feature Extraction ---")

# Extract day of week and flag weekends (Saturday = 5, Sunday = 6)
df['Day of Week'] = df['Timestamp'].dt.day_name()
df['Is_Weekend'] = df['Timestamp'].dt.dayofweek >= 5
df['Day Type'] = df['Is_Weekend'].map({True: 'Weekend', False: 'Weekday'})

# Extract Month and Map Seasons
df['Month'] = df['Timestamp'].dt.month
def assign_season(month):
    if month in [6, 7, 8]:
        return 'Summer (Peak)'
    else:
        return 'Off-Season (Spring/Autumn/Winter)'

df['Season'] = df['Month'].apply(assign_season)

# Extract Hour and Map Time Bands (Morning, Afternoon, Evening/Night)
df['Hour'] = df['Timestamp'].dt.hour
def assign_time_band(hour):
    if 6 <= hour < 12:
        return 'Morning (06:00 - 11:59)'
    elif 12 <= hour < 18:
        return 'Afternoon (12:00 - 17:59)'
    else:
        return 'Evening / Night (18:00 - 05:59)'

df['Time Band'] = df['Hour'].apply(assign_time_band)


print("\n--- 3. Running Temporal Segmentations ---")

# A. Weekday vs Weekend Efficiency Analysis
weekday_vs_weekend = df.groupby('Day Type')[['Total Activity Load', 'Operational Load Index']].mean().reset_index()
print("\n[Segment 1] Weekday vs Weekend Efficiency:\n", weekday_vs_weekend)

# B. Seasonal Utilization Patterns (Summer vs Off-Season)
seasonal_patterns = df.groupby('Season')[['Total Activity Load', 'Operational Load Index']].mean().reset_index()
print("\n[Segment 2] Seasonal Utilization Patterns:\n", seasonal_patterns)

# C. Morning, Afternoon, Evening Efficiency Band Analysis
time_band_efficiency = df.groupby('Time Band')[['Total Activity Load', 'Operational Load Index']].mean().reset_index()
print("\n[Segment 3] Time Band Efficiency Bands:\n", time_band_efficiency)


--- 2. Temporal Feature Extraction ---

--- 3. Running Temporal Segmentations ---

[Segment 1] Weekday vs Weekend Efficiency:
   Day Type  Total Activity Load  Operational Load Index
0  Weekday            80.302653                0.005559
1  Weekend           143.944778                0.009965

[Segment 2] Seasonal Utilization Patterns:
                               Season  Total Activity Load  \
0  Off-Season (Spring/Autumn/Winter)            51.642559   
1                      Summer (Peak)           206.285988   

   Operational Load Index  
0                0.003575  
1                0.014281  

[Segment 3] Time Band Efficiency Bands:
                          Time Band  Total Activity Load  \
0        Afternoon (12:00 - 17:59)           163.367057   
1  Evening / Night (18:00 - 05:59)            38.041776   
2          Morning (06:00 - 11:59)            94.754475   

   Operational Load Index  
0                0.011310  
1                0.002634  
2                0.006560  


In [13]:



print("\n--- 2. Comparative Analysis Modules ---")

# A. Peak Congestion Intervals vs Idle Intervals
# Define Peak Congestion as top 10% activity load, Idle as bottom 25%
peak_threshold = df['Total Activity Load'].quantile(0.90)
idle_threshold = df['Total Activity Load'].quantile(0.25)

peak_intervals = df[df['Total Activity Load'] >= peak_threshold]
idle_intervals = df[df['Total Activity Load'] <= idle_threshold]

print(f"Total Peak Congestion Intervals (>90th percentile): {len(peak_intervals):,}")
print(f"Total Idle Intervals (<25th percentile): {len(idle_intervals):,}")


# B. High-Cost Low-Utilization Windows
# Identify windows where sales/operations are active but passenger load is very low (e.g., high sales count relative to extremely low redemptions or vice-versa)
# Proxy for high-cost low-utilization: Intervals with low Total Activity Load despite being during operational hours
df['Year'] = df['Timestamp'].dt.year
df['Month'] = df['Timestamp'].dt.month
df['Hour'] = df['Timestamp'].dt.hour

low_utilization_windows = df[df['Total Activity Load'] <= idle_threshold]
print(f"Identified {len(low_utilization_windows):,} potential high-cost low-utilization intervals.")


# C. Trend Shifts Across Years (2015–2025)
yearly_trend = df.groupby('Year').agg({
    'Sales Count': 'sum',
    'Redemption Count': 'sum',
    'Total Activity Load': 'mean',
    'Operational Load Index': 'mean'
}).reset_index()

print("\n[Yearly Trend Shifts (2015–2025)]:\n", yearly_trend.to_string(index=False))


--- 2. Comparative Analysis Modules ---
Total Peak Congestion Intervals (>90th percentile): 26,234
Total Idle Intervals (<25th percentile): 67,327
Identified 67,327 potential high-cost low-utilization intervals.

[Yearly Trend Shifts (2015–2025)]:
  Year  Sales Count  Redemption Count  Total Activity Load  Operational Load Index
 2015      1189620            978707           144.603334                0.010011
 2016      1518428           1425779           126.687048                0.008770
 2017       682346            698861            62.495226                0.004326
 2018      1463589           1493560           119.892520                0.008300
 2019      1249725           1278505           102.481962                0.007095
 2020       366606            374546            32.591003                0.002256
 2021       782368            773040            61.646704                0.004268
 2022      1346659           1349778           104.785179                0.007254
 2023      1

In [15]:
print("\n--- Computing Project KPIs ---")

# 1. Capacity Utilization Ratio
# Measure of average ferry load efficiency relative to maximum capacity observed
max_load = df['Total Activity Load'].max()
df['Operational Load Index'] = df['Total Activity Load'] / (max_load if max_load > 0 else 1)
capacity_utilization_ratio = df['Operational Load Index'].mean() * 100


# 2. Congestion Pressure Index
# Identifies over-utilized intervals (defined as intervals exceeding the 90th percentile of activity load)
congestion_threshold = df['Total Activity Load'].quantile(0.90)
df['Is_Congested'] = df['Total Activity Load'] >= congestion_threshold
congestion_pressure_index = df['Is_Congested'].mean() * 100


# 3. Idle Capacity Percentage
# Measures under-utilization (intervals in the bottom 25% quartile of activity)
idle_threshold = df['Total Activity Load'].quantile(0.25)
df['Is_Idle'] = df['Total Activity Load'] <= idle_threshold
idle_capacity_percentage = df['Is_Idle'].mean() * 100


# 4. Peak Strain Duration
# Length of sustained high-pressure periods (average duration of consecutive congested intervals in minutes)
congestion_groups = (~df['Is_Congested']).cumsum()
strain_streaks = df['Is_Congested'].groupby(congestion_groups).sum()
avg_streak_intervals = strain_streaks[strain_streaks > 0].mean() if len(strain_streaks[strain_streaks > 0]) > 0 else 0
peak_strain_duration_minutes = avg_streak_intervals * 15  # Each interval is 15 minutes


# 5. Operational Variability Score
# Stability of utilization patterns measured using the Coefficient of Variation (Standard Deviation / Mean)
mean_load = df['Total Activity Load'].mean()
std_load = df['Total Activity Load'].std()
operational_variability_score = (std_load / mean_load) if mean_load > 0 else 0


print(40 * "=")
print("       FERRY PROJECT KPI REPORT           ")
print(40 * "=")
print(f"1. Capacity Utilization Ratio : {capacity_utilization_ratio:.2f}%")
print(f"2. Congestion Pressure Index   : {congestion_pressure_index:.2f}%")
print(f"3. Idle Capacity Percentage    : {idle_capacity_percentage:.2f}%")
print(f"4. Peak Strain Duration        : {peak_strain_duration_minutes:.1f} minutes ({avg_streak_intervals:.1f} intervals)")
print(f"5. Operational Variability     : {operational_variability_score:.2f} (Coefficient of Variation)")


--- Computing Project KPIs ---
       FERRY PROJECT KPI REPORT           
1. Capacity Utilization Ratio : 0.68%
2. Congestion Pressure Index   : 10.03%
3. Idle Capacity Percentage    : 25.74%
4. Peak Strain Duration        : 84.5 minutes (5.6 intervals)
5. Operational Variability     : 2.03 (Coefficient of Variation)
